# OP Data Preparation and Source Audit

This notebook loads Weinberg and Roth from their raw files with the saved SMILES lookups. It loads the **Gao** workbook's experimental rows **and** the three experimental supplemental CSVs, then applies the original quinone-to-hydroquinone transformation. Every source row receives a preparation status, followed by a chemistry-filter status.

The source-based candidate exports below are separate from the descriptor-aligned model53 inputs. The final section reconciles this candidate with model53 so differences in historical measurement selection remain visible. Running this notebook does not overwrite model53's training CSV or descriptor file.


## 1. Load raw sources and report preparation exclusions


In [ ]:
import os
from collections import Counter
from pathlib import Path

import pandas as pd
from rdkit import Chem
from sklearn.model_selection import GroupShuffleSplit

import sys
PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'Scripts' / 'op_source_prep.py').is_file():
    PROJECT_DIR = PROJECT_DIR / '2. Oxidation Potential'
if not (PROJECT_DIR / 'Scripts' / 'op_source_prep.py').is_file():
    raise FileNotFoundError('Run this notebook from the OP directory or its parent workspace')
SCRIPTS_DIR = PROJECT_DIR / 'Scripts'
EXTERNAL_FUNCTIONS_DIR = PROJECT_DIR.parent / 'External Functions'
PLOTTING_FUNCTIONS_DIR = EXTERNAL_FUNCTIONS_DIR / 'Plotting Functions'
for path in (SCRIPTS_DIR, PROJECT_DIR, EXTERNAL_FUNCTIONS_DIR, PLOTTING_FUNCTIONS_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
from op_source_prep import build_op_source_audit
from trace_op_sources import canonical
from dataprep_plotting import run_dataprep_plots

CURATED_DIR = PROJECT_DIR / 'Datasets/Curated Dataset(s)'
TRAINABLE_DIR = PROJECT_DIR / 'Datasets/trainable_data'
CURRENT_MODEL_FILE = TRAINABLE_DIR / 'OP_chemprop_train_file.csv'
SOURCE_AUDIT_FILE = CURATED_DIR / 'OP_source_preparation_audit.csv'
FILTER_AUDIT_FILE = CURATED_DIR / 'OP_source_filter_audit.csv'
CLEAN_OP_FILE = CURATED_DIR / 'OP_experimental_source_clean_candidate.csv'
TRAINABLE_OUTPUT = TRAINABLE_DIR / 'OP_chemprop_train_file_from_sources_candidate.csv'
RECONCILIATION_FILE = CURATED_DIR / 'OP_model53_source_reconciliation.csv'
CANDIDATE_COMPARISON_FILE = CURATED_DIR / 'OP_candidate_vs_model53.csv'
MODEL_CONDITION_DUPLICATES_FILE = CURATED_DIR / 'OP_model53_reported_condition_duplicates.csv'
UNKNOWN_CONDITION_REVIEW_FILE = CURATED_DIR / 'OP_unknown_condition_overlap_review.csv'

RANDOM_SEED = 12
RARE_ELEMENT_THRESHOLD = 0.01
APPLY_RARE_ELEMENT_FILTER = False
APPLY_IN_HOUSE_OVERRIDES = False

# Optional in-house measurements are omitted from this publication draft.
IN_HOUSE_OP_OVERRIDES_VS_SHE = {}

source_audit, OP_df_raw = build_op_source_audit()
CURATED_DIR.mkdir(parents=True, exist_ok=True)
source_audit.to_csv(SOURCE_AUDIT_FILE, index=False)

print('Original source rows by file:')
print(source_audit.groupby(['source_dataset', 'source_file']).size().to_string())
print('Source preparation status (each raw row counted once):')
print(pd.crosstab(source_audit.source_dataset, source_audit.preparation_status).to_string())
print(f'All raw file rows: {len(source_audit)}; ready for chemistry filters: {len(OP_df_raw)}')
print('Model53 reconciliation uses Weinberg z/Z +0.099 V and Y +0.158 V; the older curated export used +0.800 V.')

override_audit = []
if APPLY_IN_HOUSE_OVERRIDES:
    for name, (smiles, value) in IN_HOUSE_OP_OVERRIDES_VS_SHE.items():
        mask = OP_df_raw.SMILES.eq(canonical(smiles))
        override_audit.append({'compound': name, 'matched_rows': int(mask.sum()), 'new_OP_V': value})
        OP_df_raw.loc[mask, 'OP'] = value
    print('Optional in-house overrides:', override_audit)


## 2. Chemistry filters and measurement-level deduplication

Exact repeated *molecule + potential* rows are collapsed first. The candidate then keeps one measurement per molecule and reported condition: Weinberg solvent code, or Gao reference plus reported pH/acid note. Different Weinberg solvent codes and different Gao conditions remain separate. The accompanying Gao manuscript identifies dataset3-3 as one-electron Q/Q radical-anion measurements in DMF; those rows share one reported condition. Roth lacks solvent details in the local source CSV, so potential cross-source conflicts are retained for review. Every removed source row is recorded in the filter audit. The reconciliation section compares the candidate with the saved model53 dataset.


In [ ]:
def clean_smiles_dataset(
    df,
    output_path,
    output_file,
    audit_file,
    threshold=0.01,
    apply_rare_element_filter=False,
):
    smiles_col = None
    for candidate in ['SMILES', 'smiles']:
        if candidate in df.columns:
            smiles_col = candidate
            break
    if smiles_col is None:
        raise ValueError("No 'SMILES' or 'smiles' column found in DataFrame.")

    def canonicalize_smiles_inner(smiles):
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                return None
            return Chem.MolToSmiles(mol, canonical=True)
        except Exception:
            return None

    def is_organic(mol):
        return any(atom.GetSymbol() == 'C' for atom in mol.GetAtoms())

    def is_organometallic(mol):
        metal_atomic_numbers = {
            3,4,11,12,13,14,19,20,21,22,23,24,25,26,27,28,29,30,
            31,34,37,38,39,40,41,42,43,44,45,46,47,48,49,50,
            55,56,57,72,73,74,75,76,77,78,79,80,81,82,83,
            87,88,89
        }
        return any(atom.GetAtomicNum() in metal_atomic_numbers for atom in mol.GetAtoms())

    def is_ion(mol):
        return Chem.GetFormalCharge(mol) != 0

    def has_isotopic_label(mol):
        # Remove explicitly isotopically labeled compounds such as deuterated SMILES ([2H]).
        return any(atom.GetIsotope() > 0 for atom in mol.GetAtoms())


    def is_radical(mol):
        return any(atom.GetNumRadicalElectrons() > 0 for atom in mol.GetAtoms())

    def count_heavy_atoms(smiles):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        return mol.GetNumHeavyAtoms()

    def analyze_elements(df_local, smiles_column):
        atom_type_counts = Counter()
        molecule_atom_presence = Counter()
        for smi in df_local[smiles_column]:
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                continue
            mol = Chem.AddHs(mol)
            atom_symbols = [atom.GetSymbol() for atom in mol.GetAtoms()]
            atom_type_counts.update(atom_symbols)
            molecule_atom_presence.update(set(atom_symbols))
        return pd.DataFrame({
            'Total_Atom_Count': pd.Series(atom_type_counts),
            'Molecule_Count': pd.Series(molecule_atom_presence)
        }).fillna(0).astype(int)

    def remove_rare_elements(df_local, smiles_column, threshold_local):
        atom_summary_local = analyze_elements(df_local, smiles_column=smiles_column)
        total_molecules = len(df_local)
        if total_molecules == 0:
            return df_local.copy(), atom_summary_local, [], 0

        rare_elements = atom_summary_local[
            atom_summary_local['Molecule_Count'] / total_molecules < threshold_local
        ].index.tolist()

        mask = []
        for smi in df_local[smiles_column]:
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                mask.append(False)
                continue
            atoms = {atom.GetSymbol() for atom in mol.GetAtoms()}
            mask.append(not any(el in atoms for el in rare_elements))

        filtered_df = df_local[mask].reset_index(drop=True)
        removed_count = len(df_local) - len(filtered_df)
        return filtered_df, atom_summary_local, rare_elements, removed_count

    stats = {
        'invalid': [],
        'multi_component': [],
        'inorganic': [],
        'organometallic': [],
        'ion': [],
        'isotopic_label': [],
        'radical': [],
        'kept': [],
    }

    for _, row in df.iterrows():
        row_copy = row.copy()
        smi = row_copy[smiles_col]
        cano = canonicalize_smiles_inner(smi)
        if cano is None:
            stats['invalid'].append(row_copy)
            continue

        # Filter out multi-component SMILES (e.g., salts/mixtures like 'CCO.Cl')
        if '.' in str(cano):
            row_copy[smiles_col] = cano
            stats['multi_component'].append(row_copy)
            continue

        mol = Chem.MolFromSmiles(cano)
        if mol is None:
            stats['invalid'].append(row_copy)
            continue

        row_copy[smiles_col] = cano

        if not is_organic(mol):
            stats['inorganic'].append(row_copy)
            continue
        if is_organometallic(mol):
            stats['organometallic'].append(row_copy)
            continue
        if is_ion(mol):
            stats['ion'].append(row_copy)
            continue
        if has_isotopic_label(mol):
            stats['isotopic_label'].append(row_copy)
            continue
        if is_radical(mol):
            stats['radical'].append(row_copy)
            continue

        stats['kept'].append(row_copy)

    pre_dedup_df = pd.DataFrame(stats['kept'])
    # A supplemental copy can supply a reported condition absent from the
    # first occurrence of an otherwise identical molecule + target pair.
    pair_conditions = pre_dedup_df.groupby([smiles_col, 'OP']).reported_condition.agg(
        lambda values: tuple(sorted(set(values.dropna())))
    )
    if pair_conditions.map(len).gt(1).any():
        raise ValueError('Identical molecule + target pairs have conflicting reported conditions')
    pre_dedup_df['reported_condition'] = [
        values[0] if pd.isna(condition) and len(values := pair_conditions.get((smiles, potential), ())) == 1
        else condition
        for smiles, potential, condition in zip(
            pre_dedup_df[smiles_col], pre_dedup_df.OP, pre_dedup_df.reported_condition
        )
    ]
    pre_dedup_count = len(pre_dedup_df)
    duplicate_mask = pre_dedup_df.duplicated(subset=[smiles_col, 'OP'], keep='first')
    duplicate_rows = pre_dedup_df.loc[duplicate_mask].copy()
    dedup_df = pre_dedup_df.loc[~duplicate_mask].copy()
    dedup_removed = pre_dedup_count - len(dedup_df)

    # Only reported conditions with a usable source key can support this check.
    # Unknown solvent systems are left for review instead of being merged.
    condition_mask = (
        dedup_df.reported_condition.notna()
        & dedup_df.duplicated(subset=[smiles_col, 'reported_condition'], keep='first')
    )
    condition_duplicate_rows = dedup_df.loc[condition_mask].copy()
    dedup_df = dedup_df.loc[~condition_mask].copy()
    if not condition_duplicate_rows.empty:
        kept_conditions = dedup_df.loc[dedup_df.reported_condition.notna(), [smiles_col, 'reported_condition', 'OP']].rename(
            columns={'OP': 'kept_OP'}
        )
        condition_duplicate_rows = condition_duplicate_rows.merge(
            kept_conditions, on=[smiles_col, 'reported_condition'], how='left',
            validate='many_to_one'
        )
        condition_duplicate_rows['target_difference_V'] = (
            condition_duplicate_rows.OP - condition_duplicate_rows.kept_OP
        ).abs()
    condition_removed = len(condition_duplicate_rows)

    if apply_rare_element_filter:
        cleaned_df, atom_summary, rare_elements, rare_removed = remove_rare_elements(dedup_df, smiles_col, threshold)
    else:
        cleaned_df = dedup_df.reset_index(drop=True)
        atom_summary = analyze_elements(cleaned_df, smiles_col)
        rare_elements = []
        rare_removed = 0

    cleaned_df['non_H_atom_count'] = cleaned_df[smiles_col].apply(count_heavy_atoms)
    cleaned_df = cleaned_df.reset_index(drop=True)

    os.makedirs(output_path, exist_ok=True)
    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    cleaned_df.to_csv(output_file, index=False)
    print(f'Cleaned dataset saved to {output_file}')

    print('Filter breakdown (stepwise):')
    running = len(df)

    def print_step(label, removed):
        nonlocal running
        after = running - removed
        print(f'- {label}: removed {removed} | before {running} -> after {after}')
        running = after

    print_step('Invalid SMILES', len(stats['invalid']))
    print_step('Multi-component SMILES (contains .)', len(stats['multi_component']))
    print_step('Inorganic (no carbon)', len(stats['inorganic']))
    print_step('Organometallic', len(stats['organometallic']))
    print_step('Ions (non-zero formal charge)', len(stats['ion']))
    print_step('Isotopically labeled SMILES', len(stats['isotopic_label']))
    print_step('Radicals', len(stats['radical']))
    print_step('Duplicate molecule + potential', dedup_removed)
    print_step('Duplicate molecule + reported condition', condition_removed)
    if apply_rare_element_filter:
        print_step(f'Rare elements (<{threshold*100:.1f}% molecule frequency)', rare_removed)

    if apply_rare_element_filter:
        if rare_elements:
            print(f'Rare elements identified: {rare_elements}')
        else:
            print('Rare elements identified: none')

    print(f'Final retained rows: {len(cleaned_df)}')

    filter_report = {
        'invalid': len(stats['invalid']),
        'multi_component': len(stats['multi_component']),
        'inorganic': len(stats['inorganic']),
        'organometallic': len(stats['organometallic']),
        'ion': len(stats['ion']),
        'isotopic_label': len(stats['isotopic_label']),
        'radical': len(stats['radical']),
        'deduplicated': dedup_removed,
        'condition_deduplicated': condition_removed,
        'rare_elements_removed': rare_removed if apply_rare_element_filter else 0,
    }

    audit_frames = []
    for stage, rows in stats.items():
        if stage == 'kept' or not rows:
            continue
        frame = pd.DataFrame(rows)
        frame['filter_stage'] = stage
        audit_frames.append(frame)
    if not duplicate_rows.empty:
        frame = duplicate_rows.copy()
        frame['filter_stage'] = 'duplicate_molecule_potential'
        audit_frames.append(frame)
    if not condition_duplicate_rows.empty:
        frame = condition_duplicate_rows.copy()
        frame['filter_stage'] = 'duplicate_molecule_reported_condition'
        audit_frames.append(frame)
    if apply_rare_element_filter:
        retained_ids = set(zip(cleaned_df.source_file, cleaned_df.source_row))
        rare_rows = dedup_df.loc[[
            (file, row) not in retained_ids
            for file, row in zip(dedup_df.source_file, dedup_df.source_row)
        ]].copy()
        if not rare_rows.empty:
            rare_rows['filter_stage'] = 'rare_element'
            audit_frames.append(rare_rows)
    retained_rows = cleaned_df.copy()
    retained_rows['filter_stage'] = 'retained'
    audit_frames.append(retained_rows)
    filter_audit = pd.concat(audit_frames, ignore_index=True)
    if len(filter_audit) != len(df):
        raise ValueError('Chemistry filter audit did not account for every ready source row')
    filter_audit.to_csv(audit_file, index=False)
    print('Chemistry filter status by source:')
    print(pd.crosstab(filter_audit.source_dataset, filter_audit.filter_stage).to_string())
    print(f'Filter audit saved to {audit_file}')

    return cleaned_df, atom_summary, filter_report


In [ ]:
OP_df_canonical, atom_summary, filter_report = clean_smiles_dataset(
    OP_df_raw,
    output_path=CURATED_DIR,
    output_file=CLEAN_OP_FILE,
    audit_file=FILTER_AUDIT_FILE,
    threshold=RARE_ELEMENT_THRESHOLD,
    apply_rare_element_filter=APPLY_RARE_ELEMENT_FILTER,
)

print('Post-cleaning candidate shape:', OP_df_canonical.shape)
print('Candidate entries by source:', OP_df_canonical.source_dataset.value_counts().to_dict())

# Solvent details are absent for Roth in the local source CSV. Export rows
# whose molecules recur so unresolved cases can be reviewed before retraining.
unknown_molecules = set(OP_df_canonical.loc[
    OP_df_canonical.reported_condition.isna(), 'SMILES'
])
repeated_molecules = set(OP_df_canonical.SMILES.value_counts().loc[lambda counts: counts.gt(1)].index)
review_molecules = unknown_molecules & repeated_molecules
unknown_condition_review = OP_df_canonical.loc[
    OP_df_canonical.SMILES.isin(review_molecules)
].copy()
unknown_condition_review['condition_missing'] = unknown_condition_review.reported_condition.isna()
unknown_condition_review.to_csv(UNKNOWN_CONDITION_REVIEW_FILE, index=False)
print(f'Molecules with repeated rows and at least one unknown solvent condition: {len(review_molecules)}')
print(f'Unknown-condition overlap review saved to {UNKNOWN_CONDITION_REVIEW_FILE}')


## 3. Diagnostics and Functional Group Profiles


In [ ]:
run_dataprep_plots(
    OP_df_canonical,
    smiles_col='SMILES',
    property_col='OP',
    non_h_atom_col='non_H_atom_count',
    property_label='Oxidation Potential',
    property_units='V',
    property_hist_bins=50,
    top_n=30,
    max_per_row=6,
    include_reactive_group_distribution=False,
)


## 4. Split the source-based candidate by molecule

All measurements of a molecule stay in one split. This candidate split does not replace the saved descriptor-aligned model53 split.


In [ ]:
# Source-specific forced test compounds are omitted from this publication draft.
FORCED_TEST_SMILES = []
forced_smiles = {canonical(s) for s in FORCED_TEST_SMILES}
df_subset = OP_df_canonical[['SMILES', 'OP']].copy()
forced_test_df = df_subset.loc[df_subset.SMILES.isin(forced_smiles)].copy()
remaining_df = df_subset.loc[~df_subset.SMILES.isin(forced_smiles)].copy()

first = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, temp_idx = next(first.split(remaining_df, groups=remaining_df.SMILES))
train_df = remaining_df.iloc[train_idx].copy()
temp_df = remaining_df.iloc[temp_idx].copy()
second = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=RANDOM_SEED)
val_idx, test_idx = next(second.split(temp_df, groups=temp_df.SMILES))
val_df = temp_df.iloc[val_idx].copy()
test_df = pd.concat([temp_df.iloc[test_idx], forced_test_df], ignore_index=True)

train_df['split'] = 'train'
val_df['split'] = 'val'
test_df['split'] = 'test'
train_val_test_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
if train_val_test_df.groupby('SMILES').split.nunique().gt(1).any():
    raise ValueError('A candidate molecule appears in more than one split')
TRAINABLE_DIR.mkdir(parents=True, exist_ok=True)
train_val_test_df.to_csv(TRAINABLE_OUTPUT, index=False)
print('Source-based candidate split sizes:', train_val_test_df.split.value_counts().to_dict())
print('Unique molecules by split:', train_val_test_df.groupby('split').SMILES.nunique().to_dict())
print(f'Candidate train file: {TRAINABLE_OUTPUT}')


## 5. Reconcile the candidate with the saved model53 inputs

This matches both canonical molecule and oxidation potential. It shows which model53 records came from each source and lists clean source measurements that were not selected for the historical model. The audit identifies rows removed because of repeated reported conditions. For the remaining candidate-only rows, the original model53 selection reason is not documented. Gao reports pH/acid notes rather than a full solvent recipe, and the local Roth source lacks solvent metadata; review its overlaps before retraining. Gao dataset3-3 is identified in the accompanying manuscript as one-electron Q/Q radical-anion data in DMF, a distinct reaction from hydroquinone oxidation.


In [ ]:
model53 = pd.read_csv(CURRENT_MODEL_FILE)
model53['canonical_SMILES'] = model53.SMILES.map(canonical)
model53['rounded_OP'] = model53.OP.round(4)
source_pairs = {
    name: set(zip(part.SMILES, part.OP.round(4)))
    for name, part in OP_df_raw.groupby('source_dataset')
}
source_labels = []
for pair in zip(model53.canonical_SMILES, model53.rounded_OP):
    matches = [name for name, pairs in source_pairs.items() if pair in pairs]
    source_labels.append(matches[0] if len(matches) == 1 else
                         'unmatched' if not matches else 'ambiguous: ' + ','.join(matches))
model53['source_dataset'] = source_labels
condition_lookup = OP_df_raw.groupby(['SMILES', 'OP']).reported_condition.agg(
    lambda values: tuple(sorted(set(values.dropna())))
)
model53['reported_condition'] = [
    values[0] if len(values := condition_lookup.get((smiles, potential), ())) == 1 else None
    for smiles, potential in zip(model53.canonical_SMILES, model53.rounded_OP)
]
model_features = pd.read_csv(TRAINABLE_DIR / 'OP_chemprop_feat_file.csv')
if len(model_features) != len(model53):
    raise ValueError('Model53 feature rows do not align with training rows')
model53['solvent_class'] = model_features.filter(like='solvent_').idxmax(axis=1)
model53['acidity_class'] = model_features.filter(like='acidity_').idxmax(axis=1)
known = model53.reported_condition.notna()
model_condition_duplicates = model53.loc[
    known & model53.duplicated(['canonical_SMILES', 'reported_condition'], keep=False)
].copy()
model_condition_duplicates.to_csv(MODEL_CONDITION_DUPLICATES_FILE, index=False)
model_pairs = set(zip(model53.canonical_SMILES, model53.rounded_OP))
candidate_pairs = set(zip(OP_df_canonical.SMILES, OP_df_canonical.OP.round(4)))
model53['in_source_candidate'] = [pair in candidate_pairs for pair in
                                  zip(model53.canonical_SMILES, model53.rounded_OP)]
model53.to_csv(RECONCILIATION_FILE, index=False)
candidate_comparison = OP_df_canonical[[
    'source_dataset', 'source_file', 'source_row', 'SMILES', 'OP',
    'reported_condition'
]].copy()
candidate_comparison['in_model53'] = [pair in model_pairs for pair in
                                      zip(candidate_comparison.SMILES, candidate_comparison.OP.round(4))]
candidate_comparison.to_csv(CANDIDATE_COMPARISON_FILE, index=False)

print('Saved model53 rows by source and split:')
print(pd.crosstab(model53.source_dataset, model53.split).to_string())
print('Unique model53 molecules by source:')
print(model53.groupby('source_dataset').canonical_SMILES.nunique().to_string())
print(f'Model53 rows: {len(model53)}; unique molecule + potential pairs: {len(model_pairs)}')
print(f'Model53 repeated molecule + reported-condition groups: {model_condition_duplicates.groupby(["canonical_SMILES", "reported_condition"]).ngroups} ({len(model_condition_duplicates)} rows)')
print(f'Model53 same molecule + broad solvent class extra rows: {model53.duplicated(["canonical_SMILES", "solvent_class"]).sum()}')
print(f'Source-based cleaned candidate rows: {len(OP_df_canonical)}')
print(f'Candidate pairs absent from model53: {len(candidate_pairs - model_pairs)}')
print('Candidate-only measurements by source:', candidate_comparison.loc[~candidate_comparison.in_model53, 'source_dataset'].value_counts().to_dict())
print(f'Model53 pairs absent from candidate: {len(model_pairs - candidate_pairs)}')
filter_audit = pd.read_csv(FILTER_AUDIT_FILE)
condition_removed_pairs = set(zip(
    filter_audit.loc[filter_audit.filter_stage.eq('duplicate_molecule_reported_condition'), 'SMILES'],
    filter_audit.loc[filter_audit.filter_stage.eq('duplicate_molecule_reported_condition'), 'OP'].round(4),
))
print(f'Model53 pairs removed by reported-condition deduplication: {len((model_pairs - candidate_pairs) & condition_removed_pairs)}')
candidate_only = candidate_comparison.loc[~candidate_comparison.in_model53]
print('Candidate-only molecules absent from model53:', len(set(candidate_only.SMILES) - set(model53.canonical_SMILES)))
print(f'Model53 rows without one source match: {sum(label not in source_pairs for label in source_labels)}')
print(f'Reconciliation saved to {RECONCILIATION_FILE}')
print(f'Model53 reported-condition duplicates saved to {MODEL_CONDITION_DUPLICATES_FILE}')
print(f'Candidate-only measurements saved to {CANDIDATE_COMPARISON_FILE}')
if set(source_labels) != {'Gao', 'Weinberg', 'Roth'}:
    raise ValueError('Source audit does not fully reconcile with model53; inspect the saved audit')
if not (model_pairs - candidate_pairs).issubset(condition_removed_pairs):
    raise ValueError('Model53 contains rows missing from the raw source or chemistry audit')
